# Simplified Chinese support and reach on Steam

DATA2010 final project. **Main question:** should a Steam game support Simplified Chinese,
for which genres, and when? Reach = number of reviews.

This notebook is the whole project, from the two raw input files to the answers:

| Part | What it does | Output tables |
|---|---|---|
| Step 1 | clean the data and encode it | `pcd_games_clean` |
| Step 2 | SQ1: group similar games, with vs without Chinese | `pcd_sq1_groups` |
| Step 3 | SQ2: group similar games, added Chinese after launch vs did not | `pcd_sq2_groups` |
| Save and check | write the three tables, stop with an error if anything is wrong | |
| Part B | figures and insights for EDA, SQ1, SQ2, SQ3 (reads the saved tables only) | `report/figures/` |

Each table is saved whole as `data/processed/csv/pcd_<table>.csv` (to read in code) and
`data/processed/xlsx/pcd_<table>.xlsx` (to open and look at).

**How to run from scratch:** Python 3.12, `pip install -r requirements.txt`, then *Run All*
(or `jupyter nbconvert --to notebook --execute steam_chinese_analysis.ipynb --output executed.ipynb`).
The only inputs are the two raw CSV files, downloaded at pinned versions (backup: GitHub
release `raw-snapshots-v1`). Two runs give byte-identical CSV files.

**How to read it:** the markdown at the top of each step explains the strategy and why each
choice was made. The first line of each code cell sums up what the cell does; the comments
inside explain the code. Problems found in the real data are marked `DATA ISSUE`.

## Setup

In [ ]:
# Import the libraries and locate the project folders, so the notebook runs from any directory.
import ast                    # literal_eval: read "['English', 'French']" as a Python list
import csv                    # read the raw header row on its own
import re                     # regular expressions for text cleaning and name matching
import shutil                 # copy the downloaded file out of the download cache
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Project root = first folder, from here upwards, that contains requirements.txt.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
RAW = ROOT / "data" / "raw"                           # the two input files (not committed)
PROCESSED_CSV = ROOT / "data" / "processed" / "csv"   # output tables for code
PROCESSED_XLSX = ROOT / "data" / "processed" / "xlsx" # same tables, to open and look at
PREFIX = "pcd_"                                       # file prefix of every output table
print("repository root:", ROOT)

## Settings

Every fixed value used below, in one place. The step that uses a value explains why it was
chosen. All of them were set before any result was looked at.

In [ ]:
# Every fixed value of the analysis; the comment says which step uses it.

# --- data source: two versions of the same file, pinned by commit hash (Step 1.1)
HF_DATASET = "FronkonGames/steam-games-dataset"
HF_FILENAME = "games.csv"
REVISION_A = "52ba62abfa707a2fa31f4d147c2777a9d9fbd2da"   # snapshot A: last update of games.csv
SNAPSHOT_DATE_A = "2026-06-20"
REVISION_C = "aefdeb1ff1a739b92ac5338682cc2610a0328505"   # snapshot C: January 2024 catalogue
SNAPSHOT_DATE_C = "2024-01-11"

# --- raw column -> English snake_case name; only these columns are read (Step 1.1)
RAW_TO_CLEAN = {
    "AppID": "app_id",
    "Name": "name",
    "Release date": "release_date",
    "Price": "price",
    "Supported languages": "supported_languages",
    "Full audio languages": "full_audio_languages",
    "Positive": "positive",
    "Negative": "negative",
    "Developers": "developers",
    "Publishers": "publishers",
    "Genres": "genres",
    "Tags": "tags",
}
OPTIONAL_RAW_TO_CLEAN = {"Discount": "discount_pct"}   # only in snapshot A; "0" when missing

# --- languages (Steps 1.2 and 3)
ZH_SIMPLIFIED = "Simplified Chinese"
ZH_TRADITIONAL = "Traditional Chinese"
ENGLISH = "English"
MAJOR_LANGS = [               # Step 3: a control game must not have added any of these
    "German", "French", "Spanish - Spain", "Japanese", "Russian",
    "Korean", "Portuguese - Brazil",
]

# --- Chinese games (Step 1.2)
CN_CITIES = [
    "Beijing", "Shanghai", "Shenzhen", "Guangzhou", "Hangzhou", "Chengdu", "Nanjing",
    "Wuhan", "Xiamen", "Suzhou", "Chongqing", "Tianjin", "Changsha", "Fuzhou", "Hefei",
    "Jinan", "Qingdao", "Dalian", "Zhuhai", "Dongguan", "Xi'an",
]
CN_COMPANIES = [
    "Tencent", "NetEase", "miHoYo", "HoYoverse", "Game Science", "Perfect World",
    "X.D. Network", "XD Inc", "bilibili", "Gamera Games", "Hypergryph", "Lilith Games",
    "Seasun", "Kingsoft", "Papergames", "Paper Games", "Yostar", "Pathea", "NExT Studios",
    "TiMi Studio", "Leiting", "Giant Network", "Snail Games", "Shanda", "CMGE", "Zulong",
    "Hero Games", "Youthcat", "24 Entertainment",
]
# Legal suffixes that become separate items when a developer cell is split on commas.
COMPANY_SUFFIXES = ["inc", "ltd", "llc", "co", "corp", "limited", "gmbh", "plc", "sa", "srl", "co. ltd"]

# --- genre groups: user tag -> group; the first matching tag of a game wins (Step 1.2)
TAG_TO_GENRE = {
    "Visual Novel": "Visual Novel",
    "RPG": "RPG", "JRPG": "RPG", "CRPG": "RPG", "Action RPG": "RPG", "Party-Based RPG": "RPG",
    "Strategy": "Strategy", "Turn-Based Strategy": "Strategy", "RTS": "Strategy",
    "Grand Strategy": "Strategy", "4X": "Strategy", "Tower Defense": "Strategy",
    "Action": "Action", "Shooter": "Action", "FPS": "Action", "Platformer": "Action",
    "Hack and Slash": "Action", "Fighting": "Action",
    "Puzzle": "Puzzle", "Hidden Object": "Puzzle",
    "Casual": "Casual",
}

# --- bins (Step 1.2); each label is saved as its position 1, 2, 3, ... in the list
PRICE_BINS = [-0.001, 0.0, 4.999, 9.999, 19.999, float("inf")]    # USD, list price
PRICE_LABELS = ["free", "lt5", "5to10", "10to20", "gt20"]           # codes 1-5
YEAR_BAND_BINS = [0, 2017, 2019, 2021, 2100]
YEAR_BAND_LABELS = ["<=2017", "2018-2019", "2020-2021", "2022+"]    # codes 1-4
OTHER_LANG_BINS = [-1, 0, 4, 10_000]
OTHER_LANG_LABELS = ["0", "1-4", "5+"]                              # codes 1-3

# --- filters and thresholds
MIN_REVIEWS = 10              # Step 1.3
MIN_AGE_DAYS = 365            # Step 1.3
MIN_GAMES_PER_SIDE = 10       # Steps 2-3: a group is eligible only with this many games per side

# Step 1. Clean the data and encode it → `pcd_games_clean`

**Strategy.** Load the two raw snapshots, clean both in exactly the same way (so a game
looks the same in both), remove the rows that cannot answer the question, and encode the
binned columns. The result for snapshot A is the cleaned dataset `pcd_games_clean`, one row
per game; the cleaned snapshot C is used again in Step 3.

| Sub-step | What | Why |
|---|---|---|
| 1.1 Load | download A and C at pinned commits, keep 13 of 40 columns, English names | pinned commits make the input bytes identical on every run |
| 1.2 Clean | fix types and data issues, parse list cells, add outcome, treatment, confounders and filter flags; encode the bins | below |
| 1.3 Filter | remove Chinese games, Early Access, < 10 reviews, < 1 year old, no genre/price/year | below |
| 1.4 Build | keep the columns of the cleaned dataset | |
| 1.5 Check | show how many games each filter removed and which Chinese games were removed | to check the rules by hand |

## 1.1 Load the two raw snapshots

Snapshot A (2026-06-20, the last version of `games.csv`; later commits only update a 1 GB
`games.json`) and snapshot C (2024-01-11). Both are read from the dataset history by commit
hash, so the data never changes even though the author keeps updating it.

**DATA ISSUE.** The header of snapshot A merges two names (`DiscountDLC count`) while every
row holds two values; without a fix every later column is shifted by one. The header is
repaired before reading, and the load stops with an error if the columns still look shifted.

In [ ]:
# Functions to download the two snapshots, repair the header and read only the needed columns.

def download_snapshots() -> None:
    """Download games.csv of snapshots A and C at their pinned commits into data/raw/."""
    from huggingface_hub import hf_hub_download

    RAW.mkdir(parents=True, exist_ok=True)                 # create data/raw if missing
    for label, revision in (("a", REVISION_A), ("c", REVISION_C)):
        target = RAW / f"games_{label}.csv"
        # the files are committed with Git LFS; a clone without LFS holds a small text
        # pointer instead of the CSV, which is then replaced by the real download
        is_lfs_pointer = target.exists() and target.read_bytes()[:40].startswith(b"version https://git-lfs")
        if target.exists() and not is_lfs_pointer:         # real file already here: skip
            print(f"snapshot {label.upper()}: already in data/raw")
            continue
        cached = hf_hub_download(repo_id=HF_DATASET, filename=HF_FILENAME,
                                 revision=revision, repo_type="dataset")
        shutil.copyfile(cached, target)                    # copy out of the HF cache
        print(f"snapshot {label.upper()} ({revision[:8]}) downloaded")


def read_header(raw_path) -> list[str]:
    """Header row of a raw file, with the merged name split back into two (DATA ISSUE)."""
    with open(raw_path, newline="", encoding="utf-8") as f:
        header = next(csv.reader(f))                       # first row only
    fixed = []
    for name in header:
        # "DiscountDLC count" -> ["Discount", "DLC count"]; any other name stays as it is
        fixed += ["Discount", "DLC count"] if name == "DiscountDLC count" else [name]
    return fixed


def load_snapshot(raw_path) -> pd.DataFrame:
    """Read one raw snapshot: needed columns only, English names, every value as text."""
    names = read_header(raw_path)
    missing = [c for c in RAW_TO_CLEAN if c not in names]
    if missing:
        raise ValueError(f"{raw_path.name}: columns not found: {missing}")
    optional = [c for c in OPTIONAL_RAW_TO_CLEAN if c in names]
    raw = pd.read_csv(
        raw_path, names=names, header=None, skiprows=1,    # use the repaired header
        usecols=list(RAW_TO_CLEAN) + optional,             # read only these columns
        dtype=str, keep_default_na=False, na_values=[""],  # text; only "" counts as missing
        engine="c",
    )
    # Stop if columns are still shifted: language cells must look like "[...]" lists.
    langs = raw["Supported languages"].dropna()
    share_list_like = langs.str.startswith("[").mean() if len(langs) else 0
    if share_list_like < 0.9:
        raise ValueError(f"{raw_path.name}: columns look shifted; check read_header()")
    out = raw[list(RAW_TO_CLEAN)].rename(columns=RAW_TO_CLEAN)
    for raw_name, clean_name in OPTIONAL_RAW_TO_CLEAN.items():
        # copy the column when the snapshot has it, otherwise fill with "0"
        out[clean_name] = raw[raw_name] if raw_name in raw.columns else "0"
    return out

In [ ]:
# Run: download (if needed) and read both snapshots.
download_snapshots()
raw_a = load_snapshot(RAW / "games_a.csv")
raw_c = load_snapshot(RAW / "games_c.csv")
print(f"snapshot A: {len(raw_a):,} games x {raw_a.shape[1]} columns")
print(f"snapshot C: {len(raw_c):,} games x {raw_c.shape[1]} columns")
raw_a.head(3)

## 1.2 Clean each snapshot and add the analysis columns

No game is removed here (except rows without a usable id); every later filter only uses the
flags added here.

**Data issues fixed**
- **Sale prices.** Snapshot A was taken during a Steam sale (Hades: 7.49 = 24.99 at -70%).
  `price` becomes the list price, `sale price / (1 - discount / 100)`, so the price tier
  describes the game, not the sale. Snapshot C has no discount column; its prices are used
  as listed.
- **Junk in list cells.** Language and tag cells contain leftovers from the store pages
  (`English[b]*[/b]`, HTML tags, line breaks); they are removed before parsing.
- **Ids and dates.** Rows without an id or with a repeated id are dropped (first kept);
  unreadable release dates become missing and are dropped by the filters.

**Columns added**
- **Outcome:** `total_reviews = positive + negative` (reach).
- **Treatment:** supports Simplified Chinese. Traditional-only games are flagged: they are
  neither "with Simplified Chinese" nor "without Chinese".
- **Confounders** (characteristics that affect reach and differ between games with and
  without Chinese; Steps 2-3 hold them fixed):
  - `genre_group`: the first of the game's user tags found in the genre map. Steam orders
    tags by votes, so this is the genre players use most. Tags such as "Indie" are skipped:
    they say nothing about how much text a game has.
  - `price_tier`: list price in 5 tiers, a stand-in for budget and studio size.
  - `year_band`: release year in 4 bands (time on the store).
  - `n_other_langs`, `n_other_langs_bin`: languages other than English and Chinese, in 3 bins.
- **Chinese games** (flag; removed in 1.3). A game made in China has Chinese and Chinese
  players by default, so keeping it would mix "made for China" with "translated into
  Chinese". The data has no country column, so a game is Chinese if any rule matches:
  1. Chinese without English, in the text or in the voice-over. Chinese voice-over alone is
     not a rule: Cyberpunk 2077 and The Witcher 3 have it, and it would remove 4,233
     Western games.
  2. Every developer's name is in Chinese characters.
  3. Every developer's name contains a mainland Chinese city.
  4. Every developer is a known Chinese company (catches Black Myth: Wukong, which ships
     English). Rules 2-4 look at developers only, and need all of them to match, because
     publishers are often regional partners (bilibili publishes the Swedish Core Keeper in
     China) and big Western games list Chinese support studios (Ubisoft Shanghai on Far Cry).

**Encoding.** The three binned columns are saved as ordinal codes, the label's rank:

| Column | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| `price_tier` | free | < \$5 | \$5-10 | \$10-20 | > \$20 |
| `year_band` | <=2017 | 2018-2019 | 2020-2021 | 2022+ | |
| `n_other_langs_bin` | 0 | 1-4 | 5+ | | |

As text the labels sort in the wrong order ("10to20" before "5to10", "<=2017" after
"2022+"), which breaks tables and figure axes; codes sort correctly and can be used in
calculations that need an order. They are ranks, not amounts (tier 4 is not twice tier 2),
so they are not averaged or used as a plain number in a regression. `genre_group` is not
encoded: genres have no order, so a number would invent one.

In [ ]:
# Helper functions: clean list-like text cells, pick the genre, split developer names.

# DATA ISSUE: store-page leftovers in list cells: BBCode tags, HTML tags/entities, "*", line breaks
_TAG_JUNK = re.compile(r"\[/?[a-zA-Z]+\]|<[^>]+>|&amp;|&lt;|&gt;|\*|\r|\n|#lang_\w+")


def _word_regex(words: list[str]) -> re.Pattern:
    """Case-insensitive pattern matching any of `words` as a whole word."""
    return re.compile(r"\b(?:" + "|".join(re.escape(w) for w in words) + r")\b", re.IGNORECASE)


_CN_CITY = _word_regex(CN_CITIES)           # e.g. "Shanghai ... Co., Ltd."
_CN_COMPANY = _word_regex(CN_COMPANIES)     # e.g. "Game Science"
_CJK = re.compile(r"[一-鿿]")       # any Chinese character


def parse_list(value) -> list[str]:
    """List-like cell -> clean list of unique strings ("['A', 'B']", "A,B", junk, missing)."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []                                          # missing cell
    text = _TAG_JUNK.sub("", str(value)).strip()           # remove the junk first
    if not text or text in {"[]", "nan"}:
        return []
    items = None
    if text.startswith("["):
        try:
            items = ast.literal_eval(text)                 # "['A', 'B']" -> ['A', 'B']
        except (ValueError, SyntaxError):
            text = text.strip("[]")                        # broken list: fall back to commas
    if items is None:
        items = text.split(",")                            # "A,B" -> ['A', 'B']
    cleaned = [str(i).strip().strip("'\"").strip() for i in items]
    seen, out = set(), []
    for item in cleaned:                                   # drop empties and duplicates,
        if item and item not in seen:                      # keep the original order
            seen.add(item)
            out.append(item)
    return out


def first_genre(tags: list[str]) -> str | None:
    """Group of the first tag that appears in TAG_TO_GENRE; None if no tag matches."""
    for tag in tags:
        if tag in TAG_TO_GENRE:
            return TAG_TO_GENRE[tag]
    return None


def split_developers(text: str) -> list[str]:
    """"A, B, Inc." -> ["A", "B"]: split on commas and drop legal suffixes."""
    parts = [p.strip().strip(".").strip() for p in str(text).split(",")]
    suffixes = set(COMPANY_SUFFIXES)
    return [p for p in parts if p and p.lower().rstrip(".") not in suffixes]


def all_match(developers: list[str], pattern: re.Pattern) -> bool:
    """True if there is at least one developer and every developer matches `pattern`."""
    return bool(developers) and all(pattern.search(d) for d in developers)

In [ ]:
# Clean one snapshot and add every analysis column (outcome, treatment, confounders, flags).

def to_code(values: pd.Series, bins: list) -> pd.Series:
    """Bin numbers and return the bin's rank 1, 2, 3, ... (missing stays missing)."""
    # labels=False gives the bin position 0, 1, 2, ...; +1 makes it 1, 2, 3, ...
    return (pd.cut(values.astype("float"), bins, labels=False) + 1).astype("Int64")


def add_features(trim: pd.DataFrame, snapshot_date: str) -> pd.DataFrame:
    """Cleaned copy of one snapshot with all derived columns; only rows without an id are dropped."""
    df = trim.copy()

    # --- 1. types
    df["app_id"] = pd.to_numeric(df["app_id"], errors="coerce")        # bad id -> NaN
    df = df.dropna(subset=["app_id"]).drop_duplicates("app_id")        # DATA ISSUE: no / repeated id
    df["app_id"] = df["app_id"].astype("int64")
    for col in ("positive", "negative"):
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("int64")
    # DATA ISSUE: sale prices in A -> list price = price / (1 - discount / 100)
    df["price_at_snapshot"] = pd.to_numeric(df["price"], errors="coerce")
    discount = df["discount_pct"] if "discount_pct" in df else pd.Series(0, index=df.index)
    df["discount_pct"] = pd.to_numeric(discount, errors="coerce").fillna(0).clip(0, 99)
    df["price"] = (df["price_at_snapshot"] / (1 - df["discount_pct"] / 100)).round(2)
    # DATA ISSUE: dates come as "Aug 1, 2023" or "Aug 2023"; unreadable -> NaT (missing)
    df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce", format="mixed")
    df["release_year"] = df["release_date"].dt.year.astype("Int64")
    df["days_since_release"] = (pd.Timestamp(snapshot_date) - df["release_date"]).dt.days

    # --- 2. list-like text cells -> Python lists
    langs = df["supported_languages"].map(parse_list)
    audio = df["full_audio_languages"].map(parse_list)
    tags = df["tags"].map(parse_list)
    genres = df["genres"].map(parse_list)

    # --- 3. outcome
    df["total_reviews"] = df["positive"] + df["negative"]

    # --- 4. languages: treatment, other-languages confounder, text lists for Step 3
    chinese = {ZH_SIMPLIFIED, ZH_TRADITIONAL}
    df["has_zh_simplified"] = langs.map(lambda x: ZH_SIMPLIFIED in x)
    df["has_zh_traditional"] = langs.map(lambda x: ZH_TRADITIONAL in x)
    df["has_english"] = langs.map(lambda x: ENGLISH in x)
    df["n_other_langs"] = langs.map(lambda x: len(set(x) - chinese - {ENGLISH}))   # set difference
    df["n_other_langs_bin"] = to_code(df["n_other_langs"], OTHER_LANG_BINS)      # codes 1-3
    df["all_langs"] = langs.map(lambda x: "|".join(sorted(x)))                   # "English|French|..."

    # --- 5. Chinese games: one True/False column per rule, then "any rule matched"
    has_zh_audio = audio.map(lambda x: bool(set(x) & chinese))
    has_en_audio = audio.map(lambda x: ENGLISH in x)
    any_zh = df["has_zh_simplified"] | df["has_zh_traditional"]
    developers = df["developers"].fillna("").astype(str)
    publishers = df["publishers"].fillna("").astype(str)
    dev_lists = developers.map(split_developers)
    rules = {
        "chinese_without_english": (any_zh & ~df["has_english"]) | (has_zh_audio & ~has_en_audio),
        "developer_name_in_chinese": dev_lists.map(lambda d: all_match(d, _CJK)),
        "developer_in_chinese_city": dev_lists.map(lambda d: all_match(d, _CN_CITY)),
        "developer_known_chinese_company": dev_lists.map(lambda d: all_match(d, _CN_COMPANY)),
    }
    # (Python's re via .map, not .str.contains: with pyarrow installed pandas uses Arrow's
    # regex engine, which rejects some of these patterns.)
    rule_frame = pd.DataFrame(rules)
    df["is_cn_game"] = rule_frame.any(axis=1)                          # True if any rule matched
    # names of the matched rules joined by ";", for the check in 1.5
    df["cn_rules"] = rule_frame.apply(lambda r: ";".join(r.index[r.to_numpy()]), axis=1)
    df["makers"] = developers + " | " + publishers
    df["has_zh_audio"] = has_zh_audio

    # --- 6. other filter flags
    df["is_early_access"] = genres.map(lambda x: "Early Access" in x)
    df["is_traditional_only"] = df["has_zh_traditional"] & ~df["has_zh_simplified"]

    # --- 7. confounders: genre group and the encoded bins
    df["genre_group"] = tags.map(first_genre)
    df["price_tier"] = to_code(df["price"], PRICE_BINS)                # codes 1-5
    df["year_band"] = to_code(df["release_year"], YEAR_BAND_BINS)      # codes 1-4
    return df

In [ ]:
# Run: clean both snapshots the same way.
feat_a = add_features(raw_a, SNAPSHOT_DATE_A)
feat_c = add_features(raw_c, SNAPSHOT_DATE_C)
print(f"A: {len(feat_a):,} games, C: {len(feat_c):,} games")

## 1.3 Filter the rows

Applied in this order; each step is logged in a funnel (shown in 1.5).

| Filter | Why |
|---|---|
| drop Chinese games (any of the 4 rules) | made for China, not translated into Chinese |
| drop Early Access | unfinished games; their reviews behave differently |
| keep >= 10 reviews | below that, reach is noise; not higher (e.g. 500), because filtering hard on the outcome itself would shrink the gap being measured |
| keep games released >= 1 year before the snapshot | every game had time to collect reviews |
| keep games with a genre group, price and release year | needed to place the game in a comparison group |
| `pcd_games_clean` only: drop Traditional-Chinese-only games | neither "with Simplified Chinese" nor "without Chinese" |

In [ ]:
# Functions to apply named filters in order and record how many games survive each one.

def apply_filters(df: pd.DataFrame, steps: list[tuple[str, pd.Series]], table: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Keep rows passing every (name, mask) filter; return them and the funnel table."""
    rows = [{"table": table, "step": "start", "games_left": len(df)}]
    keep = pd.Series(True, index=df.index)
    for name, mask in steps:
        keep &= mask.reindex(df.index).fillna(False)       # missing value = fails the filter
        rows.append({"table": table, "step": name, "games_left": int(keep.sum())})
    return df[keep].copy(), pd.DataFrame(rows)


def base_filters(df: pd.DataFrame) -> list[tuple[str, pd.Series]]:
    """Filters used for both snapshots, as (name, True = keep) pairs, in order."""
    return [
        ("drop Chinese games (any of 4 rules)", ~df["is_cn_game"]),
        ("drop Early Access", ~df["is_early_access"]),
        (f"keep >= {MIN_REVIEWS} reviews", df["total_reviews"] >= MIN_REVIEWS),
        ("keep released >= 1 year before snapshot", df["days_since_release"] >= MIN_AGE_DAYS),
        ("keep games with a genre group, price and year",
         df["genre_group"].notna() & df["price"].notna() & df["release_year"].notna()),
    ]

## 1.4 Build the cleaned dataset `pcd_games_clean`

One row per game of snapshot A that passed every filter: identifiers, confounders (raw
value and code), the treatment `has_zh` (1 = Simplified Chinese, 0 = no Chinese at all) and
the outcome. `log_total_reviews` is added because review counts are very skewed (a few hits
have millions), so figures read better on a log scale.

In [ ]:
# Select the rows and columns of the cleaned dataset (snapshot A).

def build_games_clean(feat_a: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Cleaned snapshot A: one row per game after all filters, and its funnel."""
    steps = base_filters(feat_a) + [("drop Traditional-Chinese-only", ~feat_a["is_traditional_only"])]
    df, funnel = apply_filters(feat_a, steps, "games_clean")
    df["has_zh"] = df["has_zh_simplified"].astype(int)               # True/False -> 1/0
    df["log_total_reviews"] = np.log10(df["total_reviews"]).round(6) # rounded: same bytes every run
    for col in ["price_tier", "year_band", "n_other_langs_bin"]:
        df[col] = df[col].astype(int)                                # no missing left after filters
    cols = [
        "app_id", "name", "genre_group", "release_year", "year_band", "price", "price_tier",
        "n_other_langs", "n_other_langs_bin", "has_zh", "total_reviews", "log_total_reviews",
    ]
    return df[cols].sort_values("app_id").reset_index(drop=True), funnel

In [ ]:
# Run: build the cleaned dataset.
games_clean, funnel_a = build_games_clean(feat_a)
print(f"games_clean: {len(games_clean):,} games, {games_clean['has_zh'].sum():,} with Simplified Chinese")
games_clean.head()

## 1.5 Check what the cleaning removed

The funnel shows how many games are left after each filter; no single step should remove
most games. The list of removed Chinese games (biggest first) is there to spot a wrongly
removed Western game by eye. Shown only, not saved.

In [ ]:
# Show the funnel of snapshot A and the 20 biggest Chinese games removed.
cn_cols = ["app_id", "name", "makers", "cn_rules", "has_english", "has_zh_audio", "total_reviews"]
cn_excluded = (
    feat_a[feat_a["is_cn_game"] & (feat_a["total_reviews"] >= MIN_REVIEWS)][cn_cols]  # would pass the review filter
    .sort_values("total_reviews", ascending=False)                                   # biggest first
    .reset_index(drop=True)
)
print(f"{len(cn_excluded):,} Chinese games removed (>= {MIN_REVIEWS} reviews)")
display(funnel_a)
cn_excluded.head(20)

# Step 2. SQ1 groups → `pcd_sq1_groups`

**Question.** How much larger is the reach of games that support Simplified Chinese, and in
which genres?

**Strategy: compare only similar games.** Comparing all games with and without Chinese
directly would mix the effect of Chinese with that of the confounders (bigger-budget games
are both more often translated and more reviewed). So every game of `pcd_games_clean` is put
in a **comparison group** of similar games, and games with and without Chinese are compared
**only inside the same group**: there the confounders are the same on both sides, so the
remaining gap is not explained by them. This is why the confounders need no other treatment
in the analysis.

**SQ1 group = genre x price tier x release-year band x number of other languages**

| Confounder | Why it is in the group |
|---|---|
| Genre | genres differ in audience size and in how much text a game has |
| Price tier | budget and studio size drive both translation and reach |
| Release-year band | older games have had more time to collect reviews; bands rather than single years, because single years leave only 44% of games in usable groups (76% with bands) |
| Other languages | the strongest: games with Chinese have a median of 7 other languages, games without it 0 (69% vs 9% have 5+); without it SQ1 would compare "localised in many languages" with "not localised" rather than measure Chinese |

**The table `pcd_sq1_groups`**, one row per group: number of games and **median** reviews
on each side (medians, because review counts are very skewed), and the gap
`pct_diff = median with Chinese / median without - 1`. A group is `eligible` only with >= 10
games on each side; a median of fewer games moves too much with one game. Result: 360
groups, 178 eligible, covering 76% of games (87% of games with Chinese).

In [ ]:
# Build the SQ1 comparison table: one row per group, games with vs without Chinese.

# code -> label, to write readable group names ("Action | <=2017 | lt5 | 5+")
PRICE_LABEL = dict(enumerate(PRICE_LABELS, start=1))
YEAR_BAND_LABEL = dict(enumerate(YEAR_BAND_LABELS, start=1))
OTHER_LANGS_LABEL = dict(enumerate(OTHER_LANG_LABELS, start=1))


def add_sq1_group(games: pd.DataFrame) -> pd.DataFrame:
    """Copy of games with group_id = genre | year band | price tier | other languages."""
    return games.assign(group_id=(
        games["genre_group"] + " | " + games["year_band"].map(YEAR_BAND_LABEL) + " | "
        + games["price_tier"].map(PRICE_LABEL) + " | " + games["n_other_langs_bin"].map(OTHER_LANGS_LABEL)
    ))


def build_sq1_groups(games: pd.DataFrame) -> pd.DataFrame:
    """One row per group: games and median reviews with / without Chinese, gap, eligible."""
    keys = ["group_id", "genre_group", "year_band", "price_tier", "n_other_langs_bin"]
    wide = (
        games.groupby(keys + ["has_zh"])["total_reviews"]
        .agg(n="size", median="median")                    # count and median per group and side
        .unstack("has_zh")                                 # sides become columns: (stat, 0/1)
    )
    out = pd.DataFrame({
        "n_zh": wide[("n", 1)],
        "n_no_zh": wide[("n", 0)],
        "median_reviews_zh": wide[("median", 1)],
        "median_reviews_no_zh": wide[("median", 0)],
    }).reset_index()
    out[["n_zh", "n_no_zh"]] = out[["n_zh", "n_no_zh"]].fillna(0).astype(int)  # empty side -> 0
    out["pct_diff"] = (out["median_reviews_zh"] / out["median_reviews_no_zh"] - 1).round(6)
    out["eligible"] = (out["n_zh"] >= MIN_GAMES_PER_SIDE) & (out["n_no_zh"] >= MIN_GAMES_PER_SIDE)
    return out.sort_values(keys[1:]).reset_index(drop=True)   # genre, then codes: old -> new, cheap -> expensive

In [ ]:
# Run: build the SQ1 group table.
sq1_groups = build_sq1_groups(add_sq1_group(games_clean))
print(f"sq1_groups: {len(sq1_groups)} groups, {sq1_groups['eligible'].sum()} eligible")
sq1_groups.head()

# Step 3. SQ2 groups → `pcd_sq2_groups`

**Question.** Does adding Simplified Chinese after launch still increase reach, and in
which genres?

**Strategy: follow the same games over time.** Take the games that had no Chinese in
January 2024 (snapshot C) and are still on Steam in June 2026 (snapshot A), and measure how
much their reviews grew. Games that added Chinese in between (**treated**) are compared with
similar games that added nothing (**control**). The control arm is needed because every game
gains reviews over time anyway; the uplift is the extra growth of the treated games.

**3.1 The panel** (one row per game, the input of the groups; not saved):
- start from cleaned snapshot C with the same filters as Step 1 (so the starting point is
  measured the same way), keep games with no Chinese at all in C and still present in A;
- game characteristics are taken from C, because what a game looked like **before** adding
  Chinese is what is held fixed;
- drop games with 0 reviews in A (delisted or missing data: a growth of -100% is not real);
- outcome `growth = (reviews in A - reviews in C) / reviews in C`;
- `arm`: `treated` = added Simplified Chinese; `control` = added neither Chinese nor any
  major language (German, French, Spanish, Japanese, Russian, Korean, Portuguese-Brazil),
  because a game that added German and French at the same time had a localisation push of
  its own and is not a clean "did nothing"; `excluded` = added another major language only.

**3.2 SQ2 group = genre x release-year band.** Only 509 games added Chinese, so the group
can hold only the confounders that matter most for **growth**:

| Variable | In the group? | Why |
|---|---|---|
| Genre | yes | needed to answer per genre |
| Release-year band | yes | the strongest: among control games, median growth is +12% for games from <=2017 and +77% for 2022+, and treated games are about 2 years newer; without it the effect of Chinese would be overstated |
| Price tier | no | once the year band is fixed, growth is about the same in every tier except > \$20; genre x price x year leaves only 17 usable groups, 54% of treated games and 3 genres (no RPG, Strategy or Visual Novel) |
| Other languages | no | treated and control games have almost the same mix (0 / 1-4 / 5+: 62/25/13% vs 64/26/10%) |

Result: 24 groups, 16 eligible, covering 90% of treated games and all 6 genres.

**The table `pcd_sq2_groups`**, built as in Step 2 but with growth: number of treated and
control games, their median growth, and
`uplift_pts = median growth treated - median growth control` (0.1 = +10 growth points);
`eligible` with >= 10 games on each side.

In [ ]:
# Build the SQ2 panel: games without Chinese in C, still in A, with their growth and arm.

def build_sq2_panel(feat_a: pd.DataFrame, feat_c: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """One row per game of C (no Chinese, filtered as in Step 1) also present in A; and its funnel."""
    steps = base_filters(feat_c) + [
        ("keep games with no Chinese (Simplified or Traditional) in C",
         ~(feat_c["has_zh_simplified"] | feat_c["has_zh_traditional"])),
        ("keep games also present in A", feat_c["app_id"].isin(feat_a["app_id"])),
    ]
    c, funnel = apply_filters(feat_c, steps, "sq2_panel")
    c = c.set_index("app_id")
    a = feat_a.set_index("app_id").loc[c.index]            # same games, same order, from A

    p = pd.DataFrame(index=c.index)
    for col in ["name", "genre_group", "release_year", "year_band", "price_tier"]:
        p[col] = c[col]                                    # characteristics BEFORE: from C
    p["price_c"] = c["price"]
    p["reviews_c"] = c["total_reviews"]
    p["reviews_a"] = a["total_reviews"]
    p = p[p["reviews_a"] > 0].copy()                       # drop delisted games (0 reviews in A)
    a, c = a.loc[p.index], c.loc[p.index]
    p["growth"] = ((p["reviews_a"] - p["reviews_c"]) / p["reviews_c"]).round(6)

    # languages in A but not in C = added after January 2024 (set difference per game)
    new_langs = [
        set(la.split("|")) - set(lc.split("|")) - {""}
        for la, lc in zip(a["all_langs"], c["all_langs"])
    ]
    added_zh = a["has_zh_simplified"].to_numpy()           # had no Chinese in C, has it in A
    added_major = np.array([bool(n & set(MAJOR_LANGS)) for n in new_langs])
    # np.select: first condition that is True gives the label, otherwise the default
    p["arm"] = np.select([added_zh, ~added_zh & ~added_major], ["treated", "control"], default="excluded")
    for col in ["year_band", "price_tier"]:
        p[col] = p[col].astype(int)                        # no missing left after filters
    funnel = pd.concat([funnel, pd.DataFrame([
        {"table": "sq2_panel", "step": "keep games with reviews in A (not delisted)", "games_left": len(p)},
        {"table": "sq2_panel", "step": "arm = treated", "games_left": int((p["arm"] == "treated").sum())},
        {"table": "sq2_panel", "step": "arm = control", "games_left": int((p["arm"] == "control").sum())},
    ])], ignore_index=True)
    cols = ["name", "genre_group", "release_year", "year_band", "price_c", "price_tier",
            "reviews_c", "reviews_a", "growth", "arm"]
    return p[cols].reset_index().sort_values("app_id").reset_index(drop=True), funnel

In [ ]:
# Run: build the panel and show its funnel.
sq2_panel, funnel_c = build_sq2_panel(feat_a, feat_c)
print(f"sq2_panel: {len(sq2_panel):,} games")
print(sq2_panel["arm"].value_counts().to_string())
display(funnel_c)

In [ ]:
# Build the SQ2 comparison table: one row per group, treated vs control growth.

def add_sq2_group(panel: pd.DataFrame) -> pd.DataFrame:
    """Copy of the panel with group_id_sq2 = genre | year band."""
    return panel.assign(group_id_sq2=panel["genre_group"] + " | " + panel["year_band"].map(YEAR_BAND_LABEL))


def build_sq2_groups(panel: pd.DataFrame) -> pd.DataFrame:
    """One row per group: treated and control games, their median growth, uplift, eligible."""
    keys = ["group_id_sq2", "genre_group", "year_band"]
    used = panel[panel["arm"].isin(["treated", "control"])]          # "excluded" is not compared
    wide = used.groupby(keys + ["arm"])["growth"].agg(n="size", median="median").unstack("arm")
    out = pd.DataFrame({
        "n_treated": wide[("n", "treated")],
        "n_control": wide[("n", "control")],
        "median_growth_treated": wide[("median", "treated")].round(6),
        "median_growth_control": wide[("median", "control")].round(6),
    }).reset_index()
    out[["n_treated", "n_control"]] = out[["n_treated", "n_control"]].fillna(0).astype(int)
    out["uplift_pts"] = (out["median_growth_treated"] - out["median_growth_control"]).round(6)
    out["eligible"] = (out["n_treated"] >= MIN_GAMES_PER_SIDE) & (out["n_control"] >= MIN_GAMES_PER_SIDE)
    return out.sort_values(keys[1:]).reset_index(drop=True)          # genre, then old -> new

In [ ]:
# Run: build the SQ2 group table.
sq2_groups = build_sq2_groups(add_sq2_group(sq2_panel))
print(f"sq2_groups: {len(sq2_groups)} groups, {sq2_groups['eligible'].sum()} eligible")
sq2_groups

# Save and check

The three tables are written whole as csv (for code) and xlsx (to view); the xlsx files get
fixed document dates so their bytes do not change between runs. The checks then stop the
notebook with an error if anything is wrong.

In [ ]:
# Write the three output tables as csv and xlsx, prefix pcd_.
tables = {
    "games_clean": games_clean,   # Step 1: cleaned data, one row per game
    "sq1_groups": sq1_groups,     # Step 2: one row per SQ1 group
    "sq2_groups": sq2_groups,     # Step 3: one row per SQ2 group
}
FIXED_DATE = datetime(2026, 1, 1)
PROCESSED_CSV.mkdir(parents=True, exist_ok=True)
PROCESSED_XLSX.mkdir(parents=True, exist_ok=True)
for name, table in tables.items():
    table.to_csv(PROCESSED_CSV / f"{PREFIX}{name}.csv", index=False)
    with pd.ExcelWriter(PROCESSED_XLSX / f"{PREFIX}{name}.xlsx", engine="openpyxl") as writer:
        table.to_excel(writer, index=False, sheet_name=name[:31])     # sheet names max 31 chars
        writer.book.properties.created = FIXED_DATE                   # fixed metadata dates
        writer.book.properties.modified = FIXED_DATE
    print(f"{PREFIX}{name}: {len(table):,} rows x {table.shape[1]} columns")

In [ ]:
# Stop with an error if any table is wrong.

# structure: unique ids, codes in range, English snake_case column names
assert games_clean["app_id"].is_unique and sq2_panel["app_id"].is_unique
assert set(games_clean["has_zh"]) <= {0, 1}
assert set(sq2_panel["arm"]) <= {"treated", "control", "excluded"}
assert games_clean["price_tier"].between(1, 5).all() and games_clean["year_band"].between(1, 4).all()
assert games_clean["n_other_langs_bin"].between(1, 3).all()
for name, table in tables.items():
    assert all(re.fullmatch(r"[a-z0-9_]+", c) for c in table.columns), name

# saved files: the xlsx holds the same table as the csv
for name, table in tables.items():
    back = pd.read_excel(PROCESSED_XLSX / f"{PREFIX}{name}.xlsx")
    assert back.shape == table.shape and list(back.columns) == list(table.columns), name

# known games: data issues and rules behave as intended
by_name = games_clean.set_index("name")
assert by_name.loc["Hades", "price"] > 20                             # list price, not the 7.49 sale price
assert by_name.loc["Hades", "price_tier"] == PRICE_LABELS.index("gt20") + 1
for name in ["Cyberpunk 2077", "Far Cry® 5", "Core Keeper", "Hollow Knight"]:
    assert name in by_name.index, f"{name} should be kept"            # Western games kept
for name in ["Black Myth: Wukong", "Marvel Rivals", "Dyson Sphere Program"]:
    assert name in set(cn_excluded["name"]), f"{name} should be removed"  # Chinese games removed
assert sq2_panel["growth"].min() > -1                                 # no delisted game left

# group tables: every game in exactly one group, eligible groups really have >= 10 games per side
assert sq1_groups[["n_zh", "n_no_zh"]].to_numpy().sum() == len(games_clean)
assert sq2_groups[["n_treated", "n_control"]].to_numpy().sum() == (sq2_panel["arm"] != "excluded").sum()
assert (sq1_groups.loc[sq1_groups["eligible"], ["n_zh", "n_no_zh"]] >= MIN_GAMES_PER_SIDE).all().all()
assert (sq2_groups.loc[sq2_groups["eligible"], ["n_treated", "n_control"]] >= MIN_GAMES_PER_SIDE).all().all()
print("all checks passed")

# Part B. Analysis: figures and insights

Steps 1-3 did the data wrangling; Part B only reads the three saved tables, draws the
figures (saved in `report/figures/`) and states the insights. It can be run on its own.

| Section | Reads |
|---|---|
| B1 EDA | `pcd_games_clean` |
| B2 SQ1 | `pcd_sq1_groups` |
| B3 SQ2 | `pcd_sq2_groups` |
| B4 SQ3 | `pcd_sq1_groups` and `pcd_sq2_groups`, per genre |

In [ ]:
# Load the saved tables for the analysis.
def load(name: str) -> pd.DataFrame:
    return pd.read_csv(PROCESSED_CSV / f"{PREFIX}{name}.csv")

games_clean = load("games_clean")
sq1_groups = load("sq1_groups")
sq2_groups = load("sq2_groups")
FIGURES = ROOT / "report" / "figures"

## B1. EDA

## B2. SQ1: how much larger is the reach of games with Simplified Chinese, and in which genres?

## B3. SQ2: does adding Simplified Chinese after launch still increase reach, and in which genres?

## B4. SQ3: for each genre, translate at launch, later, or not a priority?